# Gemma 4 26B A4B — Sidewise FLN Question Processor (Kaggle)

Batch-processes pre-cropped question images organized by level/sub-level zip files.
**Mixture-of-Experts** — only ~4B active params per token.  
**2x T4 GPU** — model split across both GPUs via llama.cpp CUDA.

---
**Setup**  
1. Settings → Accelerator → **GPU T4 x2**  
2. Add-ons → Secrets → Add **HF_TOKEN** (Hugging Face read token)  
3. Upload zip files via **Add Data** button (e.g. `1.0.zip`). Kaggle auto-extracts them.  
4. Cell → Run All

---
**Input format:**  
Upload zip files named `{level}.{sub_level}.zip` via Add Data.
Kaggle auto-extracts them into `/kaggle/input/{level}.{sub_level}/`.
  - `1.0.zip` → `/kaggle/input/1.0/` → Level 1, Sub-level 1.0
  - `2.1.zip` → `/kaggle/input/2.1/` → Level 2, Sub-level 2.1  
Inside each zip: individual pre-cropped question images (`.png`, `.jpg`, etc.).

**Output:** Standardized Question Knowledge Base → `/kaggle/working/FLN_Knowledge_Base/`

---
**Note:** First run downloads ~17 GB model (~11 min). Cached in `/root/gguf_cache/`.

In [ ]:
import shutil, subprocess, os, time, json, re, base64, gc, hashlib
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from IPython.display import display, Image as IPyImage
from typing import Optional
shutil.rmtree("/content/", ignore_errors=True)
try:
    shutil.rmtree("/tmp/", ignore_errors=True)
    os.makedirs("/tmp/", exist_ok=True)
except: pass
subprocess.run(["pip", "cache", "purge"], capture_output=True)
total, used, free = shutil.disk_usage("/kaggle/")
print(f"Disk free: {free // (1024**3)} GB")


In [ ]:
# Cell 1: GPU check + Install deps
import torch
if not torch.cuda.is_available():
    raise SystemExit('NO GPU. Go to Settings > Accelerator > GPU, then Run all again.')
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
OUTPUT_DIR = "/content/FLN_Knowledge_Base"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Outputs: {OUTPUT_DIR}")
!pip install opencv-python -q 2>&1 | tail -1
import cv2, numpy as np
print("Ready")


In [ ]:
# Cell 2: Hugging Face Login
from huggingface_hub import login
token = os.environ.get('HF_TOKEN')
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
if not token:
    raise ValueError('HF_TOKEN not found. Set it in Add-ons > Secrets.')
login(token)
print("Logged in")


In [ ]:
# Cell 3: Load Gemma 4 26B A4B with GPU (both T4s)
import subprocess, sys, os, shutil, time
from pathlib import Path
print(f"Torch CUDA: {torch.version.cuda}")
print(f"GPUs: {torch.cuda.device_count()}")
# Clean old cached GGUFs
for f in Path("/root/gguf_cache").glob("gemma-4-26B-*.gguf"):
    print(f"Removing old: {f.name}")
    f.unlink(missing_ok=True)
sys.stdout.flush()
# Check GPU / CUDA
for cmd in ["which nvcc", "nvcc --version", "nvidia-smi --query-gpu=name --format=csv,noheader"]:
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"  {cmd.split()[0]}: {(r.stdout or '').strip()[:100] or 'not found'}")
# Determine CUDA version for wheel selection
cuda_ver = torch.version.cuda
CUDA_WHEEL_MAP = {"12.1": "cu121", "12.2": "cu122", "12.3": "cu123",
                  "12.4": "cu124", "12.5": "cu125", "12.6": "cu126",
                  "12.7": "cu124", "12.8": "cu124"}
cuda_tag = CUDA_WHEEL_MAP.get(cuda_ver, "cu124")
print(f"Using CUDA wheel tag: {cuda_tag}")
# Strategy 1: pre-built CUDA wheel from abetlen repo (no compilation)
print("Installing pre-built CUDA wheel...")
result = subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'llama-cpp-python',
    f'--extra-index-url', f'https://abetlen.github.io/llama-cpp-python/whl/{cuda_tag}',
    '--force-reinstall', '--no-cache-dir', '--upgrade'
])
if result.returncode != 0:
    # Strategy 2: build from git source (with stubs fix for cmake)
    print("Pre-built wheel not found. Building from source...")
    subprocess.run(["apt-get", "install", "-y", "-qq",
        "cmake", "ninja-build"], capture_output=True)
    stubs = "/usr/local/cuda/lib64/stubs"
    env = os.environ.copy()
    env.update({
        "CMAKE_ARGS": "-DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75",
        "FORCE_CMAKE": "1",
        "CUDACXX": "/usr/local/cuda/bin/nvcc",
        "CUDA_HOME": "/usr/local/cuda",
        "CUDA_PATH": "/usr/local/cuda",
        "CMAKE_LIBRARY_PATH": stubs,
        "CMAKE_PREFIX_PATH": "/usr/local/cuda",
    })
    subprocess.run(["ln", "-sf", f"{stubs}/libcuda.so", "/usr/lib/libcuda.so"], capture_output=True)
    subprocess.run(["ldconfig"], capture_output=True)
    result = subprocess.run([
        sys.executable, '-m', 'pip', 'install',
        'git+https://github.com/abetlen/llama-cpp-python.git',
        '--force-reinstall', '--no-cache-dir', '--upgrade', '--verbose'
    ], env=env)
if result.returncode != 0:
    raise RuntimeError("llama-cpp-python install failed. GPU required.")
print("llama-cpp-python installed")
print("llama-cpp-python installed")
from llama_cpp import Llama
from llama_cpp.llama_chat_format import Gemma4ChatHandler
CACHE_DIR = "/root/gguf_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
# Free disk space before 18 GB download
shutil.rmtree("/root/.cache/pip", ignore_errors=True)
subprocess.run(["apt-get", "clean"], capture_output=True)
total, used, free = shutil.disk_usage("/kaggle/")
print(f"Disk free: {free // (1024**3)} GB")
CHAT_HANDLER = Gemma4ChatHandler.from_pretrained(
    repo_id="unsloth/gemma-4-26B-A4B-it-GGUF",
    filename="mmproj-F16.gguf",
    local_dir=CACHE_DIR, verbose=False,
)
print("Chat handler OK")
start = time.time()
llm = Llama.from_pretrained(
    repo_id="unsloth/gemma-4-26B-A4B-it-GGUF",
    filename="gemma-4-26B-A4B-it-UD-Q4_K_XL.gguf",
    local_dir=CACHE_DIR,
    chat_handler=CHAT_HANDLER,
    n_gpu_layers=-1, n_ctx=8192,
    flash_attn=True, verbose=False,
)
t = (time.time()-start)/60
print(f"Model loaded in {t:.1f} min")


In [ ]:
# Cell 4: Preprocessing Engine
PREPROCESS_TARGET_SIZE = 1200
PREPROCESS_DESKEW_THRESHOLD = 3.0
PREPROCESS_BLUR_THRESHOLD = 80
PREPROCESS_CONTRAST_THRESHOLD = 100
PREPROCESS_CLAHE_CLIP = 2.0
PREPROCESS_CLAHE_TILE = 8
def preprocess_image(img: np.ndarray) -> tuple:
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    result = img.copy()
    applied = []
    is_blurry = lap_var < PREPROCESS_BLUR_THRESHOLD
    is_small = min(h, w) < PREPROCESS_TARGET_SIZE
    is_low_contrast = (float(gray.max()) - float(gray.min())) < PREPROCESS_CONTRAST_THRESHOLD
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLines(edges, 1, np.pi/180, 200)
    angle = 0.0
    if lines is not None:
        angles = []
        for line in lines:
            theta = line[0][1]
            deg = np.degrees(theta) - 90
            if abs(deg) < 30:
                angles.append(deg)
        if angles:
            angle = np.median(angles)
    is_skewed = abs(angle) > PREPROCESS_DESKEW_THRESHOLD
    if is_skewed:
        M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
        result = cv2.warpAffine(result, M, (w, h),
                               flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
        applied.append(f"deskew_{angle:.1f}deg")
        h, w = result.shape[:2]
    result = cv2.bilateralFilter(result, d=5, sigmaColor=50, sigmaSpace=50)
    applied.append("denoise")
    if is_blurry:
        blurred = cv2.GaussianBlur(result, (0, 0), 3.0)
        sharp = cv2.addWeighted(result, 1.5, blurred, -0.5, 0)
        sharp = np.clip(sharp, 0, 255).astype(np.uint8)
        new_var = cv2.Laplacian(cv2.cvtColor(sharp, cv2.COLOR_RGB2GRAY), cv2.CV_64F).var()
        if new_var > lap_var:
            result = sharp
            applied.append("unsharp")
    if is_low_contrast or is_blurry:
        lab = cv2.cvtColor(result, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=PREPROCESS_CLAHE_CLIP,
                                tileGridSize=(PREPROCESS_CLAHE_TILE, PREPROCESS_CLAHE_TILE))
        l = clahe.apply(l)
        result = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)
        applied.append("clahe")
    gray_result = cv2.cvtColor(result, cv2.COLOR_RGB2GRAY)
    thresholded = cv2.adaptiveThreshold(gray_result, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY, 21, 4)
    applied.append("adaptive_thresh")
    if is_small:
        scale = max(1.0, PREPROCESS_TARGET_SIZE / min(h, w))
        if scale > 1.1:
            result = cv2.resize(result, None, fx=scale, fy=scale,
                               interpolation=cv2.INTER_CUBIC)
            applied.append(f"upscale_{scale:.1f}x")
    info = {"blurry": is_blurry, "skewed": is_skewed, "small": is_small,
            "low_contrast": is_low_contrast, "lap_var": round(lap_var, 1),
            "angle": round(angle, 1), "original_size": f"{img.shape[1]}x{img.shape[0]}",
            "applied": applied}
    return result, info
def preprocess_file(image_path: str) -> tuple:
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Cannot read: {image_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return preprocess_image(img)


In [ ]:
# Cell 5: Gemma Analysis Functions
PROMPT_QUESTION = """You are FLN-Curriculum Intelligence Engine Version 1.
ROLE
You are simultaneously acting as: AI Engineer, Educational Researcher, FLN Curriculum Specialist, Child Cognitive Development Expert, Preschool Assessment Designer, Worksheet Intelligence Engine, Computer Vision Reasoning System, Educational Dataset Architect, Question Normalization Engine, Knowledge Base Builder.
MISSION
Your mission is NOT OCR. Your mission is to convert worksheet questions into standardized educational knowledge. The worksheet may originate from any publisher, organization, government department, NGO, educational website, scanned document, or printed worksheet. Ignore all publisher-specific formatting. Understand ONLY the educational meaning.
IMPORTANT
The worksheet may contain different fonts, decorative borders, watermarks, logos, different languages, different illustration styles, different layouts, tables, boxes, tracing, coloring, matching, counting, patterns, mazes, classification, sequencing activities. These differences MUST NOT affect your educational understanding.
For EVERY image independently determine:
1. Educational Concept — e.g. Counting, Matching, Tracing, Color Recognition, Alphabet Recognition, Number Recognition, Shape Recognition, Pattern Recognition, Classification, Comparison, Sorting, Sequencing, Measurement, Spatial Understanding, Logical Reasoning, Fine Motor Skills, Visual Discrimination, One-to-One Correspondence
2. Question Type — e.g. Count Objects, Circle Correct Object, Color Object, Match, Draw Line, Complete Pattern, Fill Missing Number, Trace, Cut and Paste, Tick Correct Option, Cross Incorrect Option, Compare Quantities, Arrange Objects, Find Difference, Find Same, Identify Shape, Identify Letter, Identify Number, Identify Color
3. Learning Outcome — The educational objective being taught. e.g. Count objects up to 10, Recognize circles, Identify bigger object, Recognize pattern
4. Cognitive Skills — e.g. Observation, Reasoning, Counting, Memory, Pattern Recognition, Classification, Visual Comparison, Object Recognition
5. Motor Skills — e.g. Tracing, Coloring, Drawing, Matching, Line Drawing, Writing Numbers, Writing Letters
6. Question Text — Extract the exact educational question. Ignore decorative text and publisher information.
7. Instruction — Extract only instructional text.
8. Illustration Understanding — For every illustration identify object names, categories, counts, relative positions, arrangement, and educational purpose.
9. Variables — Determine what changes if the same template is reused. e.g. Object = Apple, Count = 5, Template = Count Objects
10. Expected Answer — Determine the correct answer.
11. Difficulty — Assign a difficulty score (0–100) and rate each component from 1–5. Consider visual complexity, instruction complexity, reasoning, working memory, fine motor requirement, pattern complexity, one-to-one correspondence. NOT simply object count.
12. Educational Normalization — Normalize publisher differences. e.g. "How many apples?" / "Count the apples." / "Write the number." all become Template: Count Objects, Concept: Counting
13. Question Template — Extract reusable template. e.g. Count {Object}, Match {Shape}, Trace {Letter}, Complete {Pattern}, Circle {Correct Object}
14. Question Family — A stable identifier for the template family. e.g. MATCH_OBJECT_OUTLINE_V1, COUNT_FRUITS_V1, TRACE_LETTER_V1. This enables random paper generation without repeating the same question.
15. Confidence — Estimate confidence between 0.0 and 1.0.
RULES
Never summarize. Never invent information that is not visible. If uncertain, return null. Ignore publisher, logo, watermark, page decoration, border, color theme, brand, website. Never output markdown. Never explain reasoning. Never write paragraphs. Return ONLY valid JSON.
{
  "concept": "",
  "question_type": "",
  "question_family": "",
  "learning_outcome": "",
  "instruction": "",
  "question_text": "",
  "cognitive_skills": [],
  "motor_skills": [],
  "difficulty": {
    "score": 0,
    "visual_complexity": 0,
    "reasoning_complexity": 0,
    "instruction_complexity": 0,
    "motor_skill": 0,
    "working_memory": 0
  },
  "expected_answer": "",
  "template": "",
  "variables": {},
  "illustration": {
    "objects": [],
    "count": 0,
    "arrangement": "",
    "purpose": ""
  },
  "confidence": 0.0
}
"""
def parse_json(raw: str) -> Optional[dict]:
    cleaned = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    cleaned = re.sub(r'```json\s*|```\s*', '', cleaned).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        a, b = cleaned.find('{'), cleaned.rfind('}')
        if a != -1 and b > a:
            try:
                return json.loads(cleaned[a:b+1])
            except json.JSONDecodeError:
                pass
    return None
def analyze_with_gemma(img: np.ndarray, prompt: str) -> dict:
    _, buffer = cv2.imencode(".png", cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    b64 = base64.b64encode(buffer).decode("utf-8")
    resp = llm.create_chat_completion(
        messages=[{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
            {"type": "text", "text": prompt},
        ]}],
        max_tokens=4096, temperature=0.1,
    )
    raw = resp["choices"][0]["message"]["content"]
    parsed = parse_json(raw)
    return {"raw": raw, "parsed": parsed}


In [ ]:
# Cell 6: Normalization + Validation + Repository Logic
CONCEPT_NORMALIZATION_MAP = {
    "counting": "Counting", "count": "Counting", "count and write": "Counting",
    "number recognition": "Number Recognition", "identify the number": "Number Recognition",
    "missing number": "Missing Numbers", "what comes after": "Missing Numbers",
    "ascending": "Ascending Order", "descending": "Descending Order",
    "skip count": "Skip Counting", "addition": "Addition", "add": "Addition",
    "find the sum": "Addition", "subtraction": "Subtraction", "subtract": "Subtraction",
    "take away": "Subtraction", "word problem": "Word Problem", "story sum": "Word Problem",
    "shapes": "Shapes", "pattern": "Patterns", "match": "Matching",
    "comparison": "Comparison", "measurement": "Measurement", "money": "Money",
    "time": "Time", "fraction": "Fractions",
    "letter recognition": "Letter Recognition", "alphabet": "Letter Recognition",
    "phonics": "Phonics", "sound": "Phonics", "vocabulary": "Vocabulary",
    "reading": "Reading Comprehension", "comprehension": "Reading Comprehension",
    "sentence": "Sentence Writing", "writing practice": "Writing Practice",
    "trace": "Tracing", "tracing": "Tracing", "coloring": "Coloring",
    "classification": "Classification", "sort": "Classification",
    "logical": "Logical Reasoning", "reasoning": "Logical Reasoning",
    "review": "Review Assessment", "assessment": "Review Assessment",
}
REQUIRED_FIELDS = [
    "question_id", "level", "sub_level", "worksheet_id", "publisher",
    "concept", "learning_outcome", "question_type", "instruction",
    "question_text", "illustration", "difficulty", "expected_answer",
    "cognitive_skills", "motor_skills", "template", "variables",
    "question_family", "bounding_box", "confidence",
]
CONFIDENCE_AUTO_SAVE = 0.90
CONFIDENCE_REVIEW = 0.70
def normalize_concept(raw: str) -> str:
    raw_lower = raw.strip().lower()
    for key, val in CONCEPT_NORMALIZATION_MAP.items():
        if key in raw_lower:
            return val
    return raw
def normalize_difficulty(raw: str) -> str:
    r = raw.strip().lower()
    if r in ("easy", "e", "1", "simple"): return "easy"
    if r in ("medium", "m", "2", "intermediate"): return "medium"
    if r in ("hard", "h", "3", "difficult"): return "hard"
    return raw
def validate_record(record: dict) -> dict:
    v = dict(record)
    defaults = {
        "question_id": "unknown", "level": "", "sub_level": "",
        "worksheet_id": "", "publisher": "unknown", "concept": "",
        "learning_outcome": "", "question_type": "", "instruction": "",
        "question_text": "",
        "illustration": {"objects": [], "count": 0, "arrangement": "", "purpose": ""},
        "difficulty": {"score": 0, "visual_complexity": 0, "reasoning_complexity": 0,
                       "instruction_complexity": 0, "motor_skill": 0, "working_memory": 0},
        "expected_answer": "",
        "cognitive_skills": [], "motor_skills": [],
        "template": "", "variables": {}, "question_family": "",
        "bounding_box": {"x": 0, "y": 0, "width": 0, "height": 0},
        "confidence": 0.0,
    }
    for field in REQUIRED_FIELDS:
        if field not in v or v[field] is None:
            v[field] = defaults.get(field, "")
    if not isinstance(v.get("cognitive_skills"), list):
        v["cognitive_skills"] = []
    if not isinstance(v.get("motor_skills"), list):
        v["motor_skills"] = []
    if not isinstance(v.get("variables"), dict):
        v["variables"] = {}
    if not isinstance(v.get("illustration"), dict):
        v["illustration"] = {"objects": [], "count": 0, "arrangement": "", "purpose": ""}
    if not isinstance(v.get("bounding_box"), dict):
    if not isinstance(v.get("difficulty"), dict) or "score" not in v["difficulty"]:
        v["difficulty"] = {"score": 50, "visual_complexity": 3, "reasoning_complexity": 3,
                           "instruction_complexity": 3, "motor_skill": 3, "working_memory": 3}
        v["bounding_box"] = {"x": 0, "y": 0, "width": 0, "height": 0}
    try:
        v["confidence"] = max(0.0, min(1.0, float(v.get("confidence", 0))))
    except (ValueError, TypeError):
        v["confidence"] = 0.0
    v["_validated"] = True
    return v
all_results = []
auto_saved = 0
review_count = 0
def save_question(record: dict, output_dir: str):
    qid = record.get("question_id", "unknown")
    ws = record.get("worksheet_id", "unknown")
    path = os.path.join(output_dir, f"{ws}_{qid}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(record, f, indent=2, ensure_ascii=False)
    return path
def auto_or_review(record: dict, output_dir: str) -> str:
    global auto_saved, review_count
    conf = float(record.get("confidence", 0))
    if conf >= CONFIDENCE_AUTO_SAVE:
        save_question(record, output_dir)
        auto_saved += 1
        return "auto_saved"
    else:
        save_question(record, output_dir)
        review_count += 1
        return "review"
def build_final_question(qid, level, sub_level, ws_id, publisher, concept,
                         learning_outcome, qtype, instruction, qtext,
                         illustration, difficulty, expected_answer,
                         cognitive_skills, motor_skills, template,
                         variables, question_family, bbox, confidence) -> dict:
    return {
        "question_id": qid,
        "level": level,
        "sub_level": sub_level,
        "worksheet_id": ws_id,
        "publisher": publisher,
        "concept": concept,
        "learning_outcome": learning_outcome,
        "question_type": qtype,
        "instruction": instruction,
        "question_text": qtext,
        "illustration": illustration or {"objects": [], "count": 0, "arrangement": "", "purpose": ""},
        "difficulty": difficulty,
        "expected_answer": expected_answer,
        "cognitive_skills": cognitive_skills or [],
        "motor_skills": motor_skills or [],
        "template": template or "",
        "variables": variables or {},
        "question_family": question_family or "",
        "bounding_box": bbox or {"x": 0, "y": 0, "width": 0, "height": 0},
        "confidence": confidence,
    }


In [ ]:
# Cell 7: Sidewise Batch Processor — Main Execution
PUBLISHER = "IITRPR"  # Change as needed
RESULTS_DIR = os.path.join(OUTPUT_DIR, "questions")
os.makedirs(RESULTS_DIR, exist_ok=True)
# Find input dataset folders in /content/uploaded_zips/
# Kaggle auto-extracts zip uploads into folders here
input_folders = []
for entry in sorted(os.listdir("/content/uploaded_zips/")):
    fp = os.path.join("/content/uploaded_zips/", entry)
    if os.path.isdir(fp) and not entry.startswith("."):
        input_folders.append(entry)
if not input_folders:
    print("No input folders found in /content/uploaded_zips/.")
    print("Upload zip files via Add Data button (top right), then re-run.")
else:
    print(f"Found {len(input_folders)} dataset(s):")
    for f in input_folders:
        print(f"  {f}")
overall_start = time.time()
total_questions = 0
def calibrate_sublevel(records: list):
    """Assign relative_sublevel labels based on percentile thresholds within a sub-level."""
    scores = []
    for r in records:
        d = r.get("difficulty", {})
        if isinstance(d, dict) and isinstance(d.get("score"), (int, float)):
            scores.append(d["score"])
    if not scores:
        for r in records:
            d = r.get("difficulty", {})
            if isinstance(d, dict):
                d["relative_sublevel"] = "Medium"
        return
    if len(scores) < 3:
        for r in records:
            d = r.get("difficulty", {})
            if isinstance(d, dict):
                d["relative_sublevel"] = "Medium"
        return
    scores.sort()
    n = len(scores)
    easy_cut = scores[int(n * 0.3)]
    hard_cut = scores[int(n * 0.7)]
    for r in records:
        d = r.get("difficulty", {})
        if isinstance(d, dict) and "score" in d:
            s = d["score"]
            if s <= easy_cut:
                d["relative_sublevel"] = "Easy"
            elif s <= hard_cut:
                d["relative_sublevel"] = "Medium"
            else:
                d["relative_sublevel"] = "Hard"
for folder_name in input_folders:
    input_dir = os.path.join("/content/uploaded_zips/", folder_name)
    # folder_name acts as level.sub_level identifier (e.g. "1.0")
    parts = folder_name.split(".")
    level = parts[0] if len(parts) > 0 else ""
    sub_level = folder_name if len(parts) > 1 else ""
    print(f"\n{'='*60}")
    print(f"Processing: {folder_name}  (Level {level}, Sub-level {sub_level})")
    print(f"{'='*60}")
    # Collect images recursively from the input folder
    image_paths = []
    for ext in (".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tiff"):
        for fp in sorted(Path(input_dir).rglob(f"*{ext}")):
            image_paths.append(str(fp))
    print(f"  Images: {len(image_paths)}")
    sublevel_records = []
    retry_images = []
    def process_single_image(img_path, folder_name, level, sub_level):
        img, prep_info = preprocess_file(img_path)
        q_result = analyze_with_gemma(img, PROMPT_QUESTION)
        q_data = q_result.get("parsed")
        if not isinstance(q_data, dict):
            q_data = {}
        img_name = os.path.basename(img_path)
        qid = f"{folder_name}_{os.path.splitext(img_name)[0]}"
        concept = normalize_concept(q_data.get("concept", ""))
        qtype = normalize_concept(q_data.get("question_type", ""))
        difficulty = q_data.get("difficulty", {})
        if not isinstance(difficulty, dict) or not difficulty.get("score"):
            difficulty = {"score": 50, "visual_complexity": 3, "reasoning_complexity": 3,
                         "instruction_complexity": 3, "motor_skill": 3, "working_memory": 3}
        illustration = q_data.get("illustration", {})
        if not isinstance(illustration, dict):
            illustration = {"objects": [], "count": 0, "arrangement": "", "purpose": ""}
        record = build_final_question(
            qid=qid, level=level, sub_level=sub_level,
            ws_id=folder_name, publisher=PUBLISHER,
            concept=concept, learning_outcome=q_data.get("learning_outcome", ""),
            qtype=qtype, instruction=q_data.get("instruction", ""),
            qtext=q_data.get("question_text", ""), illustration=illustration,
            difficulty=difficulty, expected_answer=q_data.get("expected_answer", ""),
            cognitive_skills=q_data.get("cognitive_skills", []),
            motor_skills=q_data.get("motor_skills", []),
            template=q_data.get("template", ""), variables=q_data.get("variables", {}),
            question_family=q_data.get("question_family", ""),
            bbox={"x": 0, "y": 0, "width": 0, "height": 0},
            confidence=float(q_data.get("confidence", 0.7)),
        )
        return validate_record(record), difficulty
    for idx, img_path in enumerate(image_paths, 1):
        img_name = os.path.basename(img_path)
        print(f"  [{idx}/{len(image_paths)}] {img_name}", end="", flush=True)
        try:
            record, difficulty = process_single_image(img_path, folder_name, level, sub_level)
            sublevel_records.append(record)
            score = difficulty.get("score", "?")
            print(f" \u2192 {record['question_type'] or '?'} | score={score} | conf={record['confidence']:.2f}")
        except Exception as e:
            print(f" \u2192 ERROR: {e}")
            retry_images.append((img_path, idx, len(image_paths)))
    # Retry failed images once
    if retry_images:
        print(f"  Retrying {len(retry_images)} failed images...")
        still_failed = []
        for img_path, idx, total in retry_images:
            img_name = os.path.basename(img_path)
            print(f"  [retry {idx}/{total}] {img_name}", end="", flush=True)
            try:
                record, difficulty = process_single_image(img_path, folder_name, level, sub_level)
                sublevel_records.append(record)
                score = difficulty.get("score", "?")
                print(f" \u2192 {record['question_type'] or '?'} | score={score} | conf={record['confidence']:.2f}")
            except Exception as e:
                print(f" \u2192 STILL FAILED: {e}")
                still_failed.append(img_path)
        if still_failed:
            retry_dir = os.path.join(OUTPUT_DIR, "retry", folder_name)
            os.makedirs(retry_dir, exist_ok=True)
            for p in still_failed:
                shutil.copy2(p, retry_dir)
            print(f"  Copied {len(still_failed)} to {retry_dir} for manual review")
    # --- Calibration pass: assign relative_sublevel based on score percentiles ---
    calibrate_sublevel(sublevel_records)
    # Save individual files and aggregate to all_results
    for record in sublevel_records:
        decision = auto_or_review(record, RESULTS_DIR)
        all_results.append(record)
        total_questions += 1
    # Print sub-level calibration summary
    rel_counts = defaultdict(int)
    for r in sublevel_records:
        d = r.get("difficulty", {})
        if isinstance(d, dict):
            rel_counts[d.get("relative_sublevel", "unset")] += 1
    print(f"  Difficulty distribution: {dict(rel_counts)}")
elapsed = (time.time() - overall_start) / 60
# Final export
print(f"\n{'='*60}")
print("PIPELINE COMPLETE")
print(f"{'='*60}")
print(f"Total questions: {total_questions}")
print(f"Auto-saved: {auto_saved}  |  Needs review: {review_count}")
print(f"Time: {elapsed:.1f} min")
# Export knowledge base
if all_results:
    kb_path = os.path.join(OUTPUT_DIR, "knowledge_base.json")
    with open(kb_path, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)
    print(f"\nKnowledge base: {kb_path}")
    # Summary by type
    type_counts = defaultdict(int)
    for r in all_results:
        type_counts[r.get("question_type", "unknown")] += 1
    print("\nQuestions by type:")
    for t, c in sorted(type_counts.items(), key=lambda x: -x[1]):
        print(f"  {t or '?'}: {c}")
    # Summary by relative_sublevel difficulty
    rel_counts = defaultdict(int)
    for r in all_results:
        d = r.get("difficulty", {})
        if isinstance(d, dict):
            rel_counts[d.get("relative_sublevel", "unknown")] += 1
    print("\nBy difficulty (relative_sublevel):")
    for d in ["Easy", "Medium", "Hard"]:
        print(f"  {d}: {rel_counts.get(d, 0)}")
